## Задача 1. My heart will go on

Датасет **titanic** из библиотеки `Seaborn` содержит информацию о пассажирах легендарного корабля Титаник, который затонул в 1912 году после столкновения с айсбергом. Этот набор данных часто используется для обучения и тестирования алгоритмов машинного обучения, особенно в задачах бинарной классификации (выжил / не выжил).

**Описание данных**

| Поле         | Тип      | Описание |
|--------------|----------|----------|
| `survived`   | int      | Выжил (1) или не выжил (0) |
| `pclass`     | int      | Класс билета (1, 2, 3) |
| `sex`        | str      | Пол (`male`/`female`) |
| `age`        | float    | Возраст |
| `sibsp`      | int      | Количество братьев/сестёр/супругов на борту |
| `parch`      | int      | Количество родителей/детей на борту |
| `fare`       | float    | Стоимость билета |
| `embarked`   | str      | Порт посадки (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton) |
| `class`      | str      | Класс билета (`First`, `Second`, `Third`) |
| `who`        | str      | Категория: `man`, `woman` или `child` |
| `adult_male` | bool     | Является ли взрослым мужчиной |
| `deck`       | str      | Палуба |
| `embark_town`| str      | Название порта посадки |
| `alive`      | str      | Выжил (`yes`/`no`) |
| `alone`      | bool     | Путешествовал один |

**Загрузка датасета**

In [165]:
import seaborn as sns
import pandas as pd
import numpy as np

titanic_data = sns.load_dataset("titanic")
titanic_data.sample(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
357,0,2,female,38.0,0,0,13.0000,S,Second,woman,False,NaN,Southampton,no,True
569,1,3,male,32.0,0,0,7.8542,S,Third,man,True,NaN,Southampton,yes,True
89,0,3,male,24.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
5,0,3,male,NaN,0,0,8.4583,Q,Third,man,True,NaN,Queenstown,no,True
686,0,3,male,14.0,4,1,39.6875,S,Third,child,False,NaN,Southampton,no,False


**Задача**

Ниже описаны 10 небольших заданий, которые вам необходимо решить.

**Подсказка**:

В некоторых заданиях вам может быть полезен метод `value_counts`.

### Часть 1

Определите число пропущенных данных для каждого столбца таблицы `titanic_data`.

In [166]:
amount_missed = np.sum(titanic_data.isnull(), axis=0)
print(amount_missed)

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


### Часть 2

Удалите все столбцы, количество пропусков в которых превышает половину количества строк в таблице.

После того, как вы удалите все столбцы, нарушающие описанное условие, удалите все строки, количество пропусков в которых превышает половину количества столбцов.

In [167]:
half_rows = titanic_data.shape[0] // 2
titanic_data = titanic_data.dropna(axis="columns", thresh=half_rows)
half_cols = titanic_data.shape[1] // 2
titanic_data = titanic_data.dropna(axis="rows", thresh=half_cols)
print(titanic_data)

     survived  pclass     sex   age  sibsp  parch     fare embarked   class  \
0           0       3    male  22.0      1      0   7.2500        S   Third   
1           1       1  female  38.0      1      0  71.2833        C   First   
2           1       3  female  26.0      0      0   7.9250        S   Third   
3           1       1  female  35.0      1      0  53.1000        S   First   
4           0       3    male  35.0      0      0   8.0500        S   Third   
..        ...     ...     ...   ...    ...    ...      ...      ...     ...   
886         0       2    male  27.0      0      0  13.0000        S  Second   
887         1       1  female  19.0      0      0  30.0000        S   First   
888         0       3  female   NaN      1      2  23.4500        S   Third   
889         1       1    male  26.0      0      0  30.0000        C   First   
890         0       3    male  32.0      0      0   7.7500        Q   Third   

       who  adult_male  embark_town alive  alone  


### Часть 3

Если вы сделали все правильно, больше всего пропусков должно остаться в столбце `"age"` - 177. Их необходимо заполнить. Заполним пропуски следующим образом:
- Если значение столбца `"who"="man"`, пропуск необходимо заполнить медианным значением известных возрастов мужчин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="woman"`, пропуск необходимо заполнить медианным значением известных возрастов женщин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="child"`, пропуск необходимо заполнить медианным значением известных возрастов детей, округленным до ближайшего целого числа;

In [168]:
man_median = titanic_data[titanic_data["who"] == "man"]
man_median = man_median["age"].dropna()
man_median = np.round(np.median(man_median), decimals=0)
woman_median = titanic_data[titanic_data["who"] == "woman"]
woman_median = woman_median["age"].dropna()
woman_median = np.round(np.median(woman_median), decimals=0)
child_median = titanic_data[titanic_data["who"] == "child"]
child_median = child_median["age"].dropna()
child_median = np.round(np.median(child_median), decimals=0)
fill_values = pd.Series(
    data=[
        (titanic_data["who"] == "man") * man_median
        + (titanic_data["who"] == "woman") * woman_median
        + (titanic_data["who"] == "child") * child_median
    ],
    index=["age"],
)
titanic_data = titanic_data.fillna(fill_values)
print(titanic_data)

     survived  pclass     sex   age  sibsp  parch     fare embarked   class  \
0           0       3    male  22.0      1      0   7.2500        S   Third   
1           1       1  female  38.0      1      0  71.2833        C   First   
2           1       3  female  26.0      0      0   7.9250        S   Third   
3           1       1  female  35.0      1      0  53.1000        S   First   
4           0       3    male  35.0      0      0   8.0500        S   Third   
..        ...     ...     ...   ...    ...    ...      ...      ...     ...   
886         0       2    male  27.0      0      0  13.0000        S  Second   
887         1       1  female  19.0      0      0  30.0000        S   First   
888         0       3  female  30.0      1      2  23.4500        S   Third   
889         1       1    male  26.0      0      0  30.0000        C   First   
890         0       3    male  32.0      0      0   7.7500        Q   Third   

       who  adult_male  embark_town alive  alone  


### Часть 4

Удалите все строки, в которых осталось больше одного пропуска. Если вы все сделали правильно, после этого действия в таблице не должно остаться пропусков.

In [169]:
titanic_data = titanic_data.dropna(axis="rows", thresh=titanic_data.shape[1] - 1)
print(np.sum(titanic_data.isnull()))

0


### Часть 5

Определите название города, из которого отправилось больше всего пассажиров.

In [170]:
town_list = titanic_data["embark_town"]
towns = np.unique(town_list)
max_town: str = ""
max: int = 0
for town in towns:
    chosen = town_list[town_list == town]
    max = np.maximum(max, chosen.size)
    if max == chosen.size:
        max_town = town
print(max_town)

Southampton


### Часть 6

Определите процент выживших пассажиров от числа пассажиров, оставшихся в таблице после очистки данных. Ответ округлите до 2 знаков после запятой.

In [171]:
survive_data = titanic_data["survived"]
share = survive_data[survive_data == 1].size / survive_data.size
share = np.round(share * 100, decimals=2)
print(share)

38.25


### Часть 7

Определите число выживших пассажиров для каждого пункта отправления. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия пунктов отправления, а значения - число выживших пассажиров.

In [172]:
survived = titanic_data[titanic_data["survived"] == 1]
town_list = survived["embark_town"]
towns = np.unique(town_list)
town_surv_amount = pd.Series(
    data=[town_list[town_list == town].size for town in towns],
    index=towns,
)
print(town_surv_amount)

Cherbourg       93
Queenstown      30
Southampton    217
dtype: int64


### Часть 8

Определите процент выживших пассажиров в каждом классе. Значения округлите до 2 знаков после запятой. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия классов, а значения - процент выживших пассажиров.

In [173]:
class_surv_list = survived["class"]
class_list = titanic_data["class"]
classes = np.unique(class_list)
class_surv_share = pd.Series(
    data=[
        class_surv_list[class_surv_list == t_class].size / class_list[class_list == t_class].size
        for t_class in classes
    ],
    index=classes,
)
class_surv_share = np.round(class_surv_share * 100, decimals=2)
print(class_surv_share)

First     62.62
Second    47.28
Third     24.24
dtype: float64


### Часть 9

Будем считать, что пассажиры, купившие билет **НЕ МЕНЕЕ** чем за $100, считаются богатыми. Определите процент выживших среди богатых пассажиров. Ответ округлите до 2 знаков после запятой. В ответе должно получиться число. 

In [174]:
fare_surv_list = survived["fare"]
rich_surv_pers = fare_surv_list[fare_surv_list > 100]
fare_list = titanic_data["fare"]
rich_persons = fare_list[fare_list > 100]
rich_share = np.round(rich_surv_pers.size / rich_persons.size * 100, decimals=2)
print(rich_share)

73.58


### Часть 10

Определите количество детей, путешествовавших в одиночку.

In [175]:
lonely_children = titanic_data[titanic_data["alone"] == True]
lonely_children = lonely_children[titanic_data["who"] == "child"]
lonely_child_amount = lonely_children.shape[0]
print(lonely_child_amount)

6


/tmp/ipykernel_32077/245038746.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  lonely_children = lonely_children[titanic_data["who"]=="child"]


Какие выводы вы можете сделать о выживших пассажирах Титаника? 